# Tutorial - Route Planning
In this notebook, you will be given an opportunity to implement Dijkstra's search algorithm on a road network in Berkeley, California. You will then modify that algorithm using a distance heuristic to perform A* search. You will then get a chance to compare your shortest path to the mapping library's path. If they match, congratulations! 

**In this notebook, you will:**
* Implement Dijkstra's search algorithm on a road network graph.
* Implement the A* search algorithm using a Euclidean Distance heuristic on a road network graph. 

For most exercises, you are provided with a suggested outline. You are encouraged to diverge from the outline if you think there is a better, more efficient way to solve a problem.
Launch the Jupyter Notebook to begin!
We recommend that you refer to the solution only after you finish this practice exercise.

In [ ]:
# This block is used to install the necessary packages and should be run only once.
# !pip3 install osmnx
# !pip3 install networkx

In [ ]:
import osmnx as ox
import networkx as nx
import queue
import math
import priority_dict

For this tutorial, we're going to be focusing on planning in Berkeley, California, between the two nodes given below. After running the code up to and includeing the box below, you should see the output of the shortest path between the two points. Your goal is to get the same output yourself when you implement Dijkstra's and A*.

In [ ]:
# Create a graph from OpenStreetMap data for the city of Berkeley, California
map_graph = ox.graph_from_place('Berkeley, California', network_type='drive')

# Due to different versions of osmnx, the way to get the nearest node from a lat/lon coordinate has changed.
# Here we use the new version of osmnx to get the nearest node for the origin point (37.8743, -122.277).
# But if you are using an older version of osmnx, you may need to use the commented out line instead.
# origin = ox.get_nearest_node(map_graph, (37.8743, -122.277)) # this is for older versions of osmnx
# origin = ox.distance.nearest_nodes(map_graph, 37.8743, -122.277) # new version
origin = list(map_graph.nodes())[0]  # For simplicity, we just take the first node in the graph as the origin.

# Select one destination node as the last node in the graph (you can choose any other node as well)
destination = list(map_graph.nodes())[-1]

# Compute the shortest path using NetworkX's built-in Dijkstra's algorithm
# Note: NetworkX's shortest_path function is Dijkstra's algorithm when the 'weight' parameter is set to 'length'
shortest_path = nx.shortest_path(map_graph, origin, destination, weight='length')

# Visualize the graph and the shortest path
fig, ax = ox.plot_graph_route(map_graph, shortest_path)

## Dijkstra's Search
First, let's focus on Dijkstra's algorithm. As a refresher, we've included the pseudocode from the lessons.

![Dijkstra's Pseudocode](dijkstra.png)


Your goal now is to find the shortest path in the graph from the origin to the goal using Dijkstra's search. Make sure to store the optimal predecessors of each vertex in the `predecessors` dictionary, so you can retrieve the optimal path once you find the goal node in your search. Good luck!

In [ ]:
# This function follows the predecessor
# backpointers and generates the equivalent path from the
# origin as a list of vertex keys.
# Note: This function is not specific to Dijkstra's algorithm 
# and can be used for any graph search algorithm that maintains a predecessors dictionary.
# This is just a helper function to reconstruct the path from the predecessors dictionary after the search is complete.
def get_path(origin_key, goal_key, predecessors):
    key = goal_key
    path = [goal_key]
    
    while (key != origin_key):
        key = predecessors[key]
        path.insert(0, key)
        
    return path

# For a given graph, origin vertex key, and goal vertex key,
# computes the shortest path in the graph from the origin vertex
# to the goal vertex using Dijkstra's algorithm.
# Returns the shortest path as a list of vertex keys.
def dijkstras_search(origin_key, goal_key, graph):
    
    # The priority queue of open vertices we've reached.
    # Keys are the vertex keys, vals are the distances.
    open_queue = priority_dict.priority_dict({})
    
    # The dictionary of closed vertices we've processed.
    closed_dict = {}
    
    # The dictionary of predecessors for each vertex.
    predecessors = {}
    
    # Add the origin to the open queue.
    open_queue[origin_key] = 0.0

    # Iterate through the open queue, until we find the goal.
    # Each time, perform a Dijkstra's update on the queue.
    goal_found = False
    while (open_queue):
        u, u_length = open_queue.pop_smallest()        
        
        # If u is the goal, we are done.
        if u == goal_key:
            goal_found = True
            break
        
        for edge_dict in graph.out_edges([u], data=True):
            # The second element of the return tuple gives us the
            # outgoing neighbour vertex.
            v = edge_dict[1]
            
            # If v has already been processed, there is no need to
            # add it to the open queue.
            if v in closed_dict:
                continue
            
            # The third element of the return tuple contains the data.
            uv_length = edge_dict[2]['length']
            
            if v in open_queue:
                v_length = open_queue[v]
                # TODO: Fill in the correct distance for the relaxation update.
                # Hint: The distance to v through u = the distance to u + the edge weight from u to v.
                if True:  # <-- Replace True with the correct expression
                    open_queue[v] = None  # <-- Replace None with the correct expression
                    predecessors[v] = u
            else:
                # TODO: Fill in the distance to vertex v through vertex u.
                # Hint: The distance to v = the distance to u + the edge weight from u to v.
                open_queue[v] = None  # <-- Replace None with the correct expression
                predecessors[v] = u 
        
        # Close u, since all of its neighbours have been added to the
        # queue (if necessary).
        closed_dict[u] = 1
    
    # If we get through entire priority queue without finding the goal,
    # something is wrong.
    if not goal_found:
        raise ValueError("Goal not found in search.")
    
    # Construct the path from the predecessors dictionary.
    return get_path(origin_key, goal_key, predecessors)

Once these two functions have been implemented, run the box below to see if your output matches that of the library function above.

In [ ]:
path = dijkstras_search(origin, destination, map_graph)        
fig, ax = ox.plot_graph_route(map_graph, path) 

## A* Search
Next, we will use a distance heuristic to implement A* search for our map search problem. Since we are using real map data here, we will need to convert the data to a format which we can use for distance computation. Each data point has a latitude and longitude associated with it, which we then have to convert into (x, y, z) coordinates on the earth (which we will assume to be a sphere with radius 6371 km). We can then take the straight line distance between these two points as an approximation for the distance between them. Over small distances, this approximation is accurate. This is implemented in the `distance_heuristic()` function below.

In [ ]:
# Computes the Euclidean distance between two vertices.
# Assume that the earth is a sphere with radius 6371 km.
def distance_heuristic(state_key, goal_key, node_data):
    n1 = node_data[state_key]
    n2 = node_data[goal_key]

    # Get the longitude and latitude for each vertex.
    long1 = n1['x']*math.pi/180.0
    lat1 = n1['y']*math.pi/180.0
    long2 = n2['x']*math.pi/180.0
    lat2 = n2['y']*math.pi/180.0
    
    # Use a spherical approximation of the earth for
    # estimating the distance between two points.
    r = 6371000
    x1 = r*math.cos(lat1)*math.cos(long1)
    y1 = r*math.cos(lat1)*math.sin(long1)
    z1 = r*math.sin(lat1)

    x2 = r*math.cos(lat2)*math.cos(long2)
    y2 = r*math.cos(lat2)*math.sin(long2)
    z2 = r*math.sin(lat2)

    d = ((x2-x1)**2 + (y2-y1)**2 + (z2-z1)**2)**0.5
    
    return d

Now, we can use our distance heuristic to perform A* search on our map. As a refresher, we've included the A* pseudocode from Module 3 below.
![A* Pseudocode](a_star.png)
This function will be implemented in the `a_star_search()` function below.

In [ ]:
# For a given graph, origin vertex key, and goal vertex key,
# computes the shortest path in the graph from the origin vertex
# to the goal vertex using A* search. 
# Returns the shortest path as a list of vertex keys.
def a_star_search(origin_key, goal_key, graph):
    # The priority queue of open vertices we've reached.
    # Keys are the vertex keys, vals are the accumulated
    # distances plus the heuristic estimates of the distance
    # to go.
    open_queue = priority_dict.priority_dict({})
    
    # The dictionary of closed vertices we've processed.
    closed_dict = {}
    
    # The dictionary of predecessors for each vertex.
    predecessors = {}
    
    # The dictionary that stores the best cost to reach each
    # vertex found so far.
    costs = {}
    
    # Get the spatial data for each vertex as a dictionary.
    node_data = graph.nodes(True)
    
    # Add the origin to the open queue and the costs dictionary.
    costs[origin_key] = 0.0
    open_queue[origin_key] = distance_heuristic(origin_key, goal_key, node_data)

    # Iterate through the open queue, until we find the goal.
    # Each time, perform an A* update on the queue.
    goal_found = False
    while (open_queue):
        u, u_heuristic = open_queue.pop_smallest()        
        u_length = costs[u]
        
        # If u is the goal, we are done.
        if u == goal_key:
            goal_found = True
            break
        
        for edge_dict in graph.out_edges([u], data=True):
            # The second element of the return tuple gives us the
            # outgoing neighbour vertex.
            v = edge_dict[1]
            
            # If v has already been processed, there is no need to
            # add it to the open queue.
            if v in closed_dict:
                continue
            
            # The third element of the return tuple contains the data.
            uv_length = edge_dict[2]['length']
            
            if v in open_queue:
                v_length = costs[v]
                # TODO: Complete this part of the A* update for vertex v in the open queue.
                # Hint: The distance to v through u = the distance to u + the edge weight from u to v.
            else:
                # TODO: Fill in the priority value for vertex v in the open queue.
                # Hint: Same as above, f(v) = g(v) + h(v).
                costs[v] = None # <-- Replace None with the correct expression for the cost to reach v through u.
                open_queue[v] = None  # <-- Replace None with the correct expression
                predecessors[v] = u 
        
        # Close u, since all of its neighbours have been added to the
        # queue (if necessary).
        closed_dict[u] = 1
    
    # If we get through entire priority queue without finding the goal,
    # something is wrong.
    if not goal_found:
        raise ValueError("Goal not found in search.")
    
    # Construct the path from the predecessors dictionary.
    return get_path(origin_key, goal_key, predecessors)

Once this function has been implemented, run the box below to see if your output matches that of the library function at the start of the notebook.

In [ ]:
path = a_star_search(origin, destination, map_graph)        
fig, ax = ox.plot_graph_route(map_graph, path) 

Congratulations! You've now implemented two important mission planning algorithms on real map data. 

## Algorithm Comparison Analysis
Let's compare the performance and results of two algorithms.

In [ ]:
import matplotlib.pyplot as plt
import time

# Run three algorithms and record time and path length
algorithms = {}

# Helper function to calculate path length
def calculate_path_length(graph, path):
    """Calculate the total length of the path"""
    length = 0
    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        # Get edge length (may have multiple edges)
        edge_data = graph.get_edge_data(u, v)
        if edge_data:
            # If there are multiple edges, take the length of the first one
            length += list(edge_data.values())[0]['length']
    return length

# Dijkstra's algorithm
start = time.time()
dijkstra_path = dijkstras_search(origin, destination, map_graph)
dijkstra_time = time.time() - start
dijkstra_length = calculate_path_length(map_graph, dijkstra_path)
algorithms['Dijkstra'] = {'path': dijkstra_path, 'time': dijkstra_time, 'length': dijkstra_length, 'color': 'red'}

# A* algorithm
start = time.time()
astar_path = a_star_search(origin, destination, map_graph)
astar_time = time.time() - start
astar_length = calculate_path_length(map_graph, astar_path)
algorithms['A*'] = {'path': astar_path, 'time': astar_time, 'length': astar_length, 'color': 'green'}

# Print comparison results
print("="*60)
print("Algorithm Performance Comparison")
print("="*60)
for name, data in algorithms.items():
    print(f"\n{name}:")
    print(f"  Path length: {data['length']:.2f} meters")
    print(f"  Execution time: {data['time']*1000:.4f} milliseconds")
    print(f"  Number of nodes: {len(data['path'])}")
print("="*60)

In [ ]:
# Overlay all paths on one figure (differences will be visible if paths differ)
fig, ax = plt.subplots(figsize=(12, 12))

# First, plot the base map
ox.plot_graph(map_graph, ax=ax, node_size=0, edge_linewidth=0.5, 
              edge_color='gray', show=False, close=False)

# Overlay the three paths
for name, data in algorithms.items():
    # Get the edges of the path
    edges = list(zip(data['path'][:-1], data['path'][1:]))
    # Plot the path
    for u, v in edges:
        # Get edge geometry information
        if map_graph.has_edge(u, v):
            edge_data = map_graph.get_edge_data(u, v)
            if edge_data:
                for key in edge_data:
                    if 'geometry' in edge_data[key]:
                        xs, ys = edge_data[key]['geometry'].xy
                        ax.plot(xs, ys, color=data['color'], linewidth=4, 
                               alpha=0.7, label=name if u == data['path'][0] else "")
                    else:
                        x = [map_graph.nodes[u]['x'], map_graph.nodes[v]['x']]
                        y = [map_graph.nodes[u]['y'], map_graph.nodes[v]['y']]
                        ax.plot(x, y, color=data['color'], linewidth=4, 
                               alpha=0.7, label=name if u == data['path'][0] else "")

# Mark start and end points
start_node = map_graph.nodes[origin]
end_node = map_graph.nodes[destination]
ax.scatter(start_node['x'], start_node['y'], c='lime', s=200, marker='o', 
          edgecolors='black', linewidths=2, zorder=5, label='Start')
ax.scatter(end_node['x'], end_node['y'], c='red', s=200, marker='s', 
          edgecolors='black', linewidths=2, zorder=5, label='Goal')

ax.legend(fontsize=12, loc='upper right')
ax.set_title('Comparison of Two Algorithm Paths\n(Blue=NetworkX, Red=Dijkstra, Green=A*)', 
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Algorithm Search Process Visualization
Visualize the nodes explored by each algorithm during the search process:
- **Light-colored nodes**: Nodes explored by the algorithm but not included in the final path
- **Dark path**: The optimal path chosen by the algorithm

In [ ]:
# Modified Dijkstra's algorithm that returns all visited nodes and predecessor relationships
def dijkstras_search_with_history(origin_key, goal_key, graph):
    open_queue = priority_dict.priority_dict({})
    closed_dict = {}
    predecessors = {}
    visited_nodes = []  # Record all visited nodes
    
    open_queue[origin_key] = 0.0
    
    goal_found = False
    while (open_queue):
        u, u_length = open_queue.pop_smallest()
        visited_nodes.append(u)  # Record visited nodes
        
        if u == goal_key:
            goal_found = True
            break
        
        for edge_dict in graph.out_edges([u], data=True):
            v = edge_dict[1]
            
            if v in closed_dict:
                continue
            
            uv_length = edge_dict[2]['length']
            
            if v not in open_queue:
                open_queue[v] = u_length + uv_length
                predecessors[v] = u
            else:
                v_length = open_queue[v]
                if u_length + uv_length < v_length:
                    open_queue[v] = u_length + uv_length
                    predecessors[v] = u 
        
        closed_dict[u] = 1
    
    if not goal_found:
        raise ValueError("Goal not found in search.")
    
    path = get_path(origin_key, goal_key, predecessors)
    return path, visited_nodes, predecessors


# Modified A* algorithm that returns all visited nodes and predecessor relationships
def a_star_search_with_history(origin_key, goal_key, graph):
    open_queue = priority_dict.priority_dict({})
    closed_dict = {}
    predecessors = {}
    costs = {}
    visited_nodes = []  # Record all visited nodes
    
    node_data = graph.nodes(True)
    
    costs[origin_key] = 0.0
    open_queue[origin_key] = distance_heuristic(origin_key, goal_key, node_data)
    
    goal_found = False
    while (open_queue):
        u, u_heuristic = open_queue.pop_smallest()
        visited_nodes.append(u)  # Record visited nodes
        u_length = costs[u]
        
        if u == goal_key:
            goal_found = True
            break
        
        for edge_dict in graph.out_edges([u], data=True):
            v = edge_dict[1]
            
            if v in closed_dict:
                continue
            
            uv_length = edge_dict[2]['length']
            
            if v not in open_queue:
                costs[v] = u_length + uv_length
                open_queue[v] = u_length + uv_length + distance_heuristic(v, goal_key, node_data)
                predecessors[v] = u
            else:
                v_length = costs[v]
                if u_length + uv_length < v_length:
                    costs[v] = u_length + uv_length
                    open_queue[v] = u_length + uv_length + distance_heuristic(v, goal_key, node_data)
                    predecessors[v] = u 
        
        closed_dict[u] = 1
    
    if not goal_found:
        raise ValueError("Goal not found in search.")
    
    path = get_path(origin_key, goal_key, predecessors)
    return path, visited_nodes, predecessors


# Run algorithms and collect search history
dijkstra_path_h, dijkstra_visited, dijkstra_predecessors = dijkstras_search_with_history(origin, destination, map_graph)
astar_path_h, astar_visited, astar_predecessors = a_star_search_with_history(origin, destination, map_graph)
nx_path_h = nx.shortest_path(map_graph, origin, destination, weight='length')

print(f"\nDijkstra's Algorithm:")
print(f"  Nodes explored: {len(dijkstra_visited)}")
print(f"  Final path nodes: {len(dijkstra_path_h)}")

print(f"\nA* Algorithm:")
print(f"  Nodes explored: {len(astar_visited)}")
print(f"  Final path nodes: {len(astar_path_h)}")

print(f"\nEfficiency Comparison:")
print(f"  A* explored {len(astar_visited)/len(dijkstra_visited)*100:.1f}% of Dijkstra's nodes")

In [ ]:
# Visualize search process
fig, axes = plt.subplots(1, 2, figsize=(24, 12))

# Dijkstra search process visualization
ax = axes[0]
ox.plot_graph(map_graph, ax=ax, node_size=0, edge_linewidth=0.3, 
              edge_color='lightgray', show=False, close=False)

# Plot explored edges (light thin lines)
for node in dijkstra_visited:
    if node in dijkstra_predecessors and node != origin:
        pred = dijkstra_predecessors[node]
        if map_graph.has_edge(pred, node):
            edge_data = map_graph.get_edge_data(pred, node)
            for key in edge_data:
                if 'geometry' in edge_data[key]:
                    xs, ys = edge_data[key]['geometry'].xy
                    ax.plot(xs, ys, color='lightcoral', linewidth=1.5, alpha=0.3, zorder=1)
                else:
                    x = [map_graph.nodes[pred]['x'], map_graph.nodes[node]['x']]
                    y = [map_graph.nodes[pred]['y'], map_graph.nodes[node]['y']]
                    ax.plot(x, y, color='lightcoral', linewidth=1.5, alpha=0.3, zorder=1)
                break  # Only draw the first edge

# Plot all explored nodes (light color)
explored_not_in_path = [n for n in dijkstra_visited if n not in dijkstra_path_h]
x_explored = [map_graph.nodes[n]['x'] for n in explored_not_in_path]
y_explored = [map_graph.nodes[n]['y'] for n in explored_not_in_path]
ax.scatter(x_explored, y_explored, c='lightcoral', s=15, alpha=0.5, 
          zorder=2, label=f'Explored nodes ({len(explored_not_in_path)})')

# Plot final path edges (dark thick lines)
for i in range(len(dijkstra_path_h) - 1):
    u, v = dijkstra_path_h[i], dijkstra_path_h[i + 1]
    if map_graph.has_edge(u, v):
        edge_data = map_graph.get_edge_data(u, v)
        for key in edge_data:
            if 'geometry' in edge_data[key]:
                xs, ys = edge_data[key]['geometry'].xy
                ax.plot(xs, ys, color='darkred', linewidth=4, alpha=0.8, zorder=4)
            else:
                x = [map_graph.nodes[u]['x'], map_graph.nodes[v]['x']]
                y = [map_graph.nodes[u]['y'], map_graph.nodes[v]['y']]
                ax.plot(x, y, color='darkred', linewidth=4, alpha=0.8, zorder=4)

# Plot final path nodes (dark color)
x_path = [map_graph.nodes[n]['x'] for n in dijkstra_path_h]
y_path = [map_graph.nodes[n]['y'] for n in dijkstra_path_h]
ax.scatter(x_path, y_path, c='darkred', s=30, alpha=0.9, 
          zorder=5, label=f'Final path nodes ({len(dijkstra_path_h)})')

# Mark start and goal
start_node = map_graph.nodes[origin]
end_node = map_graph.nodes[destination]
ax.scatter(start_node['x'], start_node['y'], c='lime', s=300, marker='o', 
          edgecolors='black', linewidths=3, zorder=6, label='Start')
ax.scatter(end_node['x'], end_node['y'], c='red', s=300, marker='s', 
          edgecolors='black', linewidths=3, zorder=6, label='Goal')

ax.legend(fontsize=11, loc='upper right')
ax.set_title(f"Dijkstra's Algorithm Search Process\nExplored {len(dijkstra_visited)} nodes", 
            fontsize=16, fontweight='bold')

# A* search process visualization
ax = axes[1]
ox.plot_graph(map_graph, ax=ax, node_size=0, edge_linewidth=0.3, 
              edge_color='lightgray', show=False, close=False)

# Plot explored edges (light thin lines)
for node in astar_visited:
    if node in astar_predecessors and node != origin:
        pred = astar_predecessors[node]
        if map_graph.has_edge(pred, node):
            edge_data = map_graph.get_edge_data(pred, node)
            for key in edge_data:
                if 'geometry' in edge_data[key]:
                    xs, ys = edge_data[key]['geometry'].xy
                    ax.plot(xs, ys, color='lightgreen', linewidth=1.5, alpha=0.3, zorder=1)
                else:
                    x = [map_graph.nodes[pred]['x'], map_graph.nodes[node]['x']]
                    y = [map_graph.nodes[pred]['y'], map_graph.nodes[node]['y']]
                    ax.plot(x, y, color='lightgreen', linewidth=1.5, alpha=0.3, zorder=1)
                break  # Only draw the first edge

# Plot all explored nodes (light color)
explored_not_in_path = [n for n in astar_visited if n not in astar_path_h]
x_explored = [map_graph.nodes[n]['x'] for n in explored_not_in_path]
y_explored = [map_graph.nodes[n]['y'] for n in explored_not_in_path]
ax.scatter(x_explored, y_explored, c='lightgreen', s=15, alpha=0.5, 
          zorder=2, label=f'Explored nodes ({len(explored_not_in_path)})')

# Plot final path edges (dark thick lines)
for i in range(len(astar_path_h) - 1):
    u, v = astar_path_h[i], astar_path_h[i + 1]
    if map_graph.has_edge(u, v):
        edge_data = map_graph.get_edge_data(u, v)
        for key in edge_data:
            if 'geometry' in edge_data[key]:
                xs, ys = edge_data[key]['geometry'].xy
                ax.plot(xs, ys, color='darkgreen', linewidth=4, alpha=0.8, zorder=4)
            else:
                x = [map_graph.nodes[u]['x'], map_graph.nodes[v]['x']]
                y = [map_graph.nodes[u]['y'], map_graph.nodes[v]['y']]
                ax.plot(x, y, color='darkgreen', linewidth=4, alpha=0.8, zorder=4)

# Plot final path nodes (dark color)
x_path = [map_graph.nodes[n]['x'] for n in astar_path_h]
y_path = [map_graph.nodes[n]['y'] for n in astar_path_h]
ax.scatter(x_path, y_path, c='darkgreen', s=30, alpha=0.9, 
          zorder=5, label=f'Final path nodes ({len(astar_path_h)})')

# Mark start and goal
ax.scatter(start_node['x'], start_node['y'], c='lime', s=300, marker='o', 
          edgecolors='black', linewidths=3, zorder=6, label='Start')
ax.scatter(end_node['x'], end_node['y'], c='red', s=300, marker='s', 
          edgecolors='black', linewidths=3, zorder=6, label='Goal')

ax.legend(fontsize=11, loc='upper right')
ax.set_title(f'A* Algorithm Search Process\nExplored {len(astar_visited)} nodes', 
            fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Direct comparison of search efficiency between two algorithms
fig, ax = plt.subplots(figsize=(14, 12))

# Plot base map
ox.plot_graph(map_graph, ax=ax, node_size=0, edge_linewidth=0.3, 
              edge_color='lightgray', show=False, close=False)

# Edges explored by Dijkstra only
dijkstra_only_nodes = set(dijkstra_visited) - set(astar_visited)
for node in dijkstra_only_nodes:
    if node in dijkstra_predecessors and node != origin:
        pred = dijkstra_predecessors[node]
        if map_graph.has_edge(pred, node):
            edge_data = map_graph.get_edge_data(pred, node)
            for key in edge_data:
                if 'geometry' in edge_data[key]:
                    xs, ys = edge_data[key]['geometry'].xy
                    ax.plot(xs, ys, color='orange', linewidth=1, alpha=0.2, zorder=1)
                else:
                    x = [map_graph.nodes[pred]['x'], map_graph.nodes[node]['x']]
                    y = [map_graph.nodes[pred]['y'], map_graph.nodes[node]['y']]
                    ax.plot(x, y, color='orange', linewidth=1, alpha=0.2, zorder=1)
                break

# Edges explored by A* only
astar_only_nodes = set(astar_visited) - set(dijkstra_visited)
for node in astar_only_nodes:
    if node in astar_predecessors and node != origin:
        pred = astar_predecessors[node]
        if map_graph.has_edge(pred, node):
            edge_data = map_graph.get_edge_data(pred, node)
            for key in edge_data:
                if 'geometry' in edge_data[key]:
                    xs, ys = edge_data[key]['geometry'].xy
                    ax.plot(xs, ys, color='green', linewidth=1, alpha=0.2, zorder=1)
                else:
                    x = [map_graph.nodes[pred]['x'], map_graph.nodes[node]['x']]
                    y = [map_graph.nodes[pred]['y'], map_graph.nodes[node]['y']]
                    ax.plot(x, y, color='green', linewidth=1, alpha=0.2, zorder=1)
                break

# Edges explored by both algorithms
both_visited_nodes = set(dijkstra_visited) & set(astar_visited)
for node in both_visited_nodes:
    if node in dijkstra_predecessors and node != origin and node not in dijkstra_path_h:
        pred = dijkstra_predecessors[node]
        if map_graph.has_edge(pred, node):
            edge_data = map_graph.get_edge_data(pred, node)
            for key in edge_data:
                if 'geometry' in edge_data[key]:
                    xs, ys = edge_data[key]['geometry'].xy
                    ax.plot(xs, ys, color='red', linewidth=1, alpha=0.3, zorder=2)
                else:
                    x = [map_graph.nodes[pred]['x'], map_graph.nodes[node]['x']]
                    y = [map_graph.nodes[pred]['y'], map_graph.nodes[node]['y']]
                    ax.plot(x, y, color='red', linewidth=1, alpha=0.3, zorder=2)
                break

# Nodes explored only by A*
astar_only = set(astar_visited) - set(dijkstra_visited)
if astar_only:
    x_astar = [map_graph.nodes[n]['x'] for n in astar_only]
    y_astar = [map_graph.nodes[n]['y'] for n in astar_only]
    ax.scatter(x_astar, y_astar, c='green', s=10, alpha=0.4, 
              zorder=3, label=f'A* only ({len(astar_only)})')

# Nodes explored by both algorithms
both_visited = set(dijkstra_visited) & set(astar_visited)
both_not_path = [n for n in both_visited if n not in dijkstra_path_h]
if both_not_path:
    x_both = [map_graph.nodes[n]['x'] for n in both_not_path]
    y_both = [map_graph.nodes[n]['y'] for n in both_not_path]
    ax.scatter(x_both, y_both, c='red', s=10, alpha=0.5, 
              zorder=4, label=f'Both explored ({len(both_not_path)})')

# Plot final path
for i in range(len(dijkstra_path_h) - 1):
    u, v = dijkstra_path_h[i], dijkstra_path_h[i + 1]
    if map_graph.has_edge(u, v):
        edge_data = map_graph.get_edge_data(u, v)
        for key in edge_data:
            if 'geometry' in edge_data[key]:
                xs, ys = edge_data[key]['geometry'].xy
                ax.plot(xs, ys, color='blue', linewidth=5, alpha=0.8, zorder=4)
            else:
                x = [map_graph.nodes[u]['x'], map_graph.nodes[v]['x']]
                y = [map_graph.nodes[u]['y'], map_graph.nodes[v]['y']]
                ax.plot(x, y, color='blue', linewidth=5, alpha=0.8, zorder=4)

# Plot path nodes
x_path = [map_graph.nodes[n]['x'] for n in dijkstra_path_h]
y_path = [map_graph.nodes[n]['y'] for n in dijkstra_path_h]
ax.scatter(x_path, y_path, c='blue', s=40, alpha=0.9, 
          zorder=5, edgecolors='white', linewidths=1, label=f'Optimal path ({len(dijkstra_path_h)} nodes)')

# Mark start and goal
start_node = map_graph.nodes[origin]
end_node = map_graph.nodes[destination]
ax.scatter(start_node['x'], start_node['y'], c='lime', s=400, marker='o', 
          edgecolors='black', linewidths=3, zorder=6, label='Start')
ax.scatter(end_node['x'], end_node['y'], c='red', s=400, marker='s', 
          edgecolors='black', linewidths=3, zorder=6, label='Goal')

ax.legend(fontsize=12, loc='upper right')
ax.set_title(f'Algorithm Search Efficiency Comparison\nDijkstra: {len(dijkstra_visited)} nodes | A*: {len(astar_visited)} nodes | Efficiency gain: {(1-len(astar_visited)/len(dijkstra_visited))*100:.1f}%', 
            fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"  Dijkstra explored {len(dijkstra_visited)} nodes")
print(f"  A* explored only {len(astar_visited)} nodes")
print(f"  A* reduced exploration by {len(dijkstra_visited) - len(astar_visited)} nodes")